In [1]:
# %%
import os
import sys
import json
import re # <-- Added
from pathlib import Path

# --- Make sure your utils are imported for pathing ---
file_path = os.getcwd()
project_root = os.path.dirname(os.path.dirname(file_path))
print(f"Project Root: {project_root}")
if project_root not in sys.path:
    sys.path.append(project_root)

from dotenv import load_dotenv
import chromadb
from langchain_core.embeddings import Embeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_ollama import OllamaEmbeddings
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma 
from langchain_core.documents import Document

# --- Import your project's agents and utils ---
from crunch_llm.code.agents import BasicAgent 
from crunch_llm.code.utils import initialize_local_data


Project Root: /oscar/data/gk/jdtoscan/LLM/Crunch_LLM


/users/jdtoscan/.conda/envs/Agent_Hub/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
analyzer_system_prompt = """
You are an expert code analyst. Your task is to analyze the provided Python class definition and extract its name and high-level purpose into a structured JSON format.

Your final output must be a single, valid JSON object. Do not add any explanatory text outside the JSON.

### Fields to Extract

1.  **`function_name`**: The name of the Python class being defined.
    * *Examples*: `Polynomial_Embedding`, `eMLP`, `RBF_KAN_layer`, `KKAN`.

2.  **`purpose`**: Infer the primary *categorical* purpose of this class. Use one of the following categories:
    * `Feature Expansion` (e.g., for `Polynomial_Embedding`, `Random_Fourier_Embedding`)
    * `Architecture Enhancement` (e.g., for `WN_layer`, `AdaptiveResNet`, `eMLP`)
    * `KAN Layer` (e.g., for `Cheby_KAN_layer`, `RBF_KAN_layer`, `Polynomial_grid_KAN_layer`)
    * `KKAN Architecture` (e.g., for `KKAN`, `get_Psi`)
    * `Unknown` (if it doesn't fit)

---

### Example 1: Feature Expansion

*Input Code Snippet (partial):*
```python
class Polynomial_Embedding(nn.Module):   
    degree: int
    step: int = 1 
    polynomial_type: str ='T'
    def setup(self):
        self.T_funcs = [globals()[f"self.polynomial_typei"] for i in range(0, self.degree+1, self.step)]
    @nn.compact
    def __call__(self, X):
        C_n = self.param('c_i', nn.initializers.ones, (len(self.T_funcs)))
        C_n_T_n = jnp.hstack([C_n[i] / (i + 1) * self.T_funcs[i](X) for i in range(len(self.T_funcs))])
        return C_n_T_n
```

*Your Required JSON Output:*
```json
{{
  "function_name": "Polynomial_Embedding",
  "purpose": "Feature Expansion"
}}
```

---

### Example 2: KAN Layer

*Input Code Snippet (partial):*
```python
class RBF_KAN_layer(nn.Module):
    out_dim: int
    degree: int
    grid_min: float = -2.0
    grid_max: float = 2.0
    def setup(self):
        self.num_centers = self.degree
        self.sigma = (self.grid_max - self.grid_min) / (self.num_centers - 1)
    @nn.compact
    def __call__(self, X):
        # ... (RBF logic) ...
        output = jnp.einsum('bin,ino->bo', RBF_n, C_n)
        return output
```

*Your Required JSON Output:*
```json
{{
  "function_name": "RBF_KAN_layer",
  "purpose": "KAN Layer"
}}
```

---

### Example 3: Architecture Enhancement

*Input Code Snippet (partial):*
```python
class eMLP(nn.Module):
    layers: Sequence[int]
    activation: Callable = nn.relu
    degree: int=5
    @nn.compact
    def __call__(self, x):
        X = Polynomial_Embedding_Layer(degree=self.degree)(x)
        H = nn.activation.tanh(WN_layer(self.layers[0], kernel_init=init)(X))
        for feat in self.layers[1:-1]:
            H = AdaptiveResNet(out_features=feat)(H) 
        H = Polynomial_Embedding_Layer(degree=self.degree)(H)
        H = WN_layer(self.layers[-1], kernel_init=init)(H)
        return H
```

*Your Required JSON Output:*
```json
{{
  "function_name": "eMLP",
  "purpose": "Architecture Enhancement"
}}
```
"""

In [4]:
my_database_path = "/users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/database_Modules_ollama"
COLLECTION_NAME = 'Layers'

vector_store = initialize_local_data(db_path=my_database_path, provider= 'ollama',collection_name=COLLECTION_NAME)

print("Initializing Layer Analyzer Agent...")
analyzer_agent = BasicAgent(
    model='gemini-2.5-flash', 
    model_provider='google',
    system_prompt=analyzer_system_prompt, # Use the new prompt
    verbose=False
)
target_file = '/users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/codes/SciML/Modules/layers.py'

all_results = {}
print(f"\nStarting analysis of: {target_file}")

filename = os.path.basename(target_file)
file_path = target_file


Initializing with Ollama. Model: bge-m3:567m
✅ Vector store initialized successfully from path: /users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/database_Modules_ollama
   Provider: ollama, Model: bge-m3:567m, Collection: Layers
Initializing Layer Analyzer Agent...

Starting analysis of: /users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/codes/SciML/Modules/layers.py


In [4]:

try:
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        code_content = f.read()
    line_count = code_content.count('\n') + 1
    print(f"Processing: {filename} ({line_count} lines)...")

    # -----------------------------------------------------
    # --- 3. NEW SPLITTING LOGIC
    # -----------------------------------------------------
    print("  - Splitting code by '%% [markdown]' delimiter...")

    split_pattern = r'(^# %%\s*\[markdown\].*?)(?=# %% \[markdown\]|$)'
    
    # We use re.findall to get a list of (markdown_chunk, code_chunk)
    # This is safer than re.split and trying to pair them up
    
    # This pattern finds the markdown, then the code *after* it
    parts = re.split(split_pattern, code_content, flags=re.DOTALL | re.MULTILINE)
    
    cleaned_chunks = []
    
    # Handle the "prologue" (imports)
    if parts[0].strip():
        cleaned_chunks.append({
            "description": "File Prologue / Imports",
            "code": parts[0].strip(),
            "is_prologue": True
        })
        
    # Loop over the split parts, taking 2 at a time
    # (This pairs the markdown delimiter with its code content)
    for i in range(1, len(parts), 2):
        if i + 1 < len(parts):
            markdown_content = parts[i].strip()
            code_part = parts[i+1].strip()
            
            # Clean the markdown delimiter
            # "# %% [markdown]\n# # Layers..." -> "# # Layers..."
            description = re.sub(r'^# %%\s*\[markdown\]\s*', '', markdown_content)
            
            if code_part: # Only add if there is code
                cleaned_chunks.append({
                    "description": description,
                    "code": code_part,
                    "is_prologue": False
                })
        
    num_chunks = len(cleaned_chunks)
    print(f"  - Found {num_chunks} chunks (prologue + function blocks).")

    # -----------------------------------------------------
    # --- 4. PREPARE METADATA AND LOOP
    # -----------------------------------------------------
    base_metadata = {
        'source_file': file_path,
        'filename': filename,
        'type': 'code-notebook-layer-doc'
    }
    
    comment = '#' # It's a .py file
    docs_to_add = []
    
    # Loop through each chunk (markdown + code pair)
    for i, chunk in enumerate(cleaned_chunks):
        part_num = i + 1
        code_part = chunk["code"]
        description_part = chunk["description"]
        
        print(f"  - Processing chunk {part_num}/{num_chunks}...")
        
        analysis_result = {}
        summary_string = "" 
        if chunk["is_prologue"]:
            print("    - Skipping agent for prologue.")
            analysis_result = {
                "function_name": "Prologue",
                "purpose": "Imports and Setup"
            }
        else:
            try:
                print("    - Running Analyzer Agent on code chunk...")
                agent_result = analyzer_agent.run(code_part)
                summary_string = agent_result['output'].strip()
                
                # --- 6. JSON PARSING *INSIDE* THE LOOP ---
                # Clean up markdown code fences if agent adds them
                if summary_string.startswith("```json"):
                    summary_string = re.sub(r"^```json\n|```$", "", summary_string, flags=re.MULTILINE)
                    
                analysis_result = json.loads(summary_string)
                print(f"    - Analysis: {analysis_result}")
                
            except Exception as e:
                # Now the 'summary_string' variable is guaranteed to exist
                print(f"    - Warning: Could not parse agent JSON summary '{summary_string}'. Error: {e}")
                analysis_result = {
                    "function_name": "Unknown",
                    "purpose": "Unknown"
                }
        
        # --- 7. Create metadata for this specific chunk ---
        chunk_metadata = base_metadata.copy()
        chunk_metadata['part'] = f"{part_num}/{num_chunks}"
        # Use first line of markdown as the simple description
        chunk_metadata['cell_description'] = description_part.split('\n', 1)[0].lstrip('# ').strip() 
        chunk_metadata.update(analysis_result) # Add analysis to metadata
        
        analysis_str = json.dumps(analysis_result, indent=2).replace('{', '').replace('}', '').replace('"', '')

        # --- 8. Create the full header for this chunk ---
        # We will embed the *full* markdown description in the header
        header = f"""{comment} CODE ANALYSIS (PART {part_num} of {num_chunks})
{comment} FUNCTION: {analysis_result.get('function_name', 'N/A')}
{comment} PURPOSE: {analysis_result.get('purpose', 'N/A')}
{comment} =======================================================
{comment} =======================================================\n\n"""
        
        enhanced_content = header + code_part
        
        doc = Document(page_content=enhanced_content, metadata=chunk_metadata)
        docs_to_add.append(doc)

    # --- 9. Add all documents for this file at once ---
    if docs_to_add:
        vector_store.add_documents(docs_to_add)
        
    print(f"✅ Ingested: {filename} ({num_chunks} chunks)")
    all_results[file_path] = {"chunks_created": num_chunks}

except Exception as e:
    print(f"--- UNHANDLED ERROR analyzing {filename}: {e} ---")
    all_results[file_path] = {"error": str(e)}

# --- 10. Save and Display Results ---
output_file = 'full_layer_analysis.json'
print(f"\nAnalysis complete! ✅ Saving all results to {output_file}...")
with open(output_file, 'w') as f:
    json.dump(all_results, f, indent=4)
print("Done.")

Processing: layers.py (802 lines)...
  - Splitting code by '%% [markdown]' delimiter...
  - Found 16 chunks (prologue + function blocks).
  - Processing chunk 1/16...
    - Running Analyzer Agent on code chunk...
    - Analysis: {'function_name': None, 'purpose': 'Unknown'}
  - Processing chunk 2/16...
    - Running Analyzer Agent on code chunk...
    - Analysis: {'function_name': 'Polynomial_Embedding', 'purpose': 'Feature Expansion'}
  - Processing chunk 3/16...
    - Running Analyzer Agent on code chunk...
    - Analysis: {'function_name': 'Random_Fourier_Embedding', 'purpose': 'Feature Expansion'}
  - Processing chunk 4/16...
    - Running Analyzer Agent on code chunk...
    - Analysis: {'function_name': 'Periodic_Fourier_Features', 'purpose': 'Feature Expansion'}
  - Processing chunk 5/16...
    - Running Analyzer Agent on code chunk...
    - Analysis: {'function_name': 'WN_layer', 'purpose': 'Architecture Enhancement'}
  - Processing chunk 6/16...
    - Running Analyzer Agent on 

In [5]:

# Assuming 'vector_store' is your initialized Chroma object
print("Fetching all documents from the vector store...")

# The .get() method returns a dictionary with IDs, documents, metadatas, etc.
# We specify what information we want to retrieve.
all_entries = vector_store.get(
    include=["metadatas", "documents"] # You can also include "embeddings"
)

# Let's print the first 5 documents found
num_entries = len(all_entries['ids'])
print(f"Found {num_entries} total entries.\n")

for i in range(min(5, num_entries)):
    doc_id = all_entries['ids'][i]
    metadata = all_entries['metadatas'][i]
    content = all_entries['documents'][i]
    
    print(f"--- Entry {i+1} ---")
    print(f"ID: {doc_id}")
    print(f"Source File: {metadata.get('source_file', 'N/A')}")
    print(f"Content Preview: {content[:500]}...") # Print first 150 chars
    print("-" * 20)


Fetching all documents from the vector store...
Found 16 total entries.

--- Entry 1 ---
ID: 52d0a223-7aa1-47a6-ae6e-0de1b81b38bc
Source File: /users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/codes/SciML/Modules/layers.py
Content Preview: # CODE ANALYSIS (PART 1 of 16)
# FUNCTION: None
# PURPOSE: Unknown
# =======================================================
# =======================================================

# # Layers
# This is the documentation for the Layers module imported as  
# 
# ```python
# from Crunch.Models.layers import *
# ```

# %%
import os
import sys
import os
import time
import jax
from jax import vmap
import jax.numpy as jnp
import optax
import matplotlib.pyplot as plt
from tqdm import trange
from jax ...
--------------------
--- Entry 2 ---
ID: 37c9d81f-0b2b-42ec-9061-4f72440c4e1f
Source File: /users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/codes/SciML/Modules/layers.py
Content Preview: # CODE ANALYSIS (PART 2 of 16)
# FUNCTION: Polynomial_Embe

In [6]:
import chromadb
import os
from pathlib import Path
import sys

# --- Make sure your utils are imported for pathing ---
# (This isn't strictly necessary but good for consistency)
file_path = os.getcwd()
project_root = os.path.dirname(file_path)
if project_root not in sys.path:
    sys.path.append(project_root)

my_collection_name = COLLECTION_NAME

print(f"Attempting to clear elements from collection: '{my_collection_name}'")
print(f"From database path: {my_database_path}")

try:
    # --- 2. INITIALIZE THE CLIENT ---
    client = chromadb.PersistentClient(path=my_database_path)
    
    # --- 3. GET THE COLLECTION ---
    collection = client.get_collection(name=my_collection_name)
    
    # --- 4. GET ALL ELEMENT IDS ---
    # We get all items. include=[] is efficient as we only need IDs.
    all_items = collection.get(include=[]) 
    all_ids = all_items['ids']
    
    if not all_ids:
        print(f"\nInfo: Collection '{my_collection_name}' is already empty.")
    else:
        # --- 5. DELETE ALL ELEMENTS BY ID ---
        print(f"\nFound {len(all_ids)} elements. Deleting all elements...")
        collection.delete(ids=all_ids)
        print(f"✅ Success: All {len(all_ids)} elements have been deleted from '{my_collection_name}'.")
        print("The collection itself still exists.")

except chromadb.errors.CollectionNotFoundError:
    print(f"\nInfo: Collection '{my_collection_name}' does not exist. Nothing to delete.")
except Exception as e:
    print(f"\nAn error occurred: {e}")
    print("Please check the path and ChromaDB server status.")



Attempting to clear elements from collection: 'Layers'
From database path: /users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/database_Modules_ollama

Found 16 elements. Deleting all elements...
✅ Success: All 16 elements have been deleted from 'Layers'.
The collection itself still exists.
